# 54장 · SQL 매개변수 바인딩과 트랜잭션

SQL 매개변수는 쿼리의 자리표시자에 값을 별도로 전달합니다. 트랜잭션에서는 변경을 확정하기 전에 되돌릴 수 있습니다.

초안 문서를 저장하고 소유자와 문서 번호를 조건으로 수정합니다. 변경 직후의 조회값을 commit 또는 rollback 이후의 값과 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 53장의 결과 열기

같은 폴더에 내려받은 노트북을 여세요. 먼저 my-course/agent-53.json를 읽습니다. 파일이 없으면 시작 셀의 완성 예시로 연습할 수 있습니다.

In [ ]:
from pathlib import Path
import json
folder = Path.cwd()
if (folder.name == "daily" or (folder.name.startswith("chapter") and folder.name[7:].isdigit())):
    folder = folder.parent
work = folder / "my-course"
work.mkdir(exist_ok=True)
source_file = work / "agent-53.json"
if source_file.exists():
    previous = json.loads(source_file.read_text(encoding="utf-8"))
else:
    previous = {'record': {'owner': 'alice', 'id': 'loan-01', 'text': '대출 기간은 7일입니다.'}}
print("내 파일" if source_file.exists() else "완성 예시", previous["record"]["text"])

완성 예시 대출 기간은 7일입니다.


## WHERE에 문서 ID와 소유자 조건 결합

강의 원자료와 같은 문서 ID로 작은 SQLite 메모리 표를 준비합니다. 쿼리 문자열의 ?는 SQLite용 값 자리입니다. 입력값을 SQL 문장에 직접 이어 붙이지 않고 별도 튜플로 넘깁니다. PostgreSQL의 psycopg에서는 이 자리를 %s로 씁니다.


### 같은 문서 ID로 소유자만 바꾼 두 조회

get_title에 같은 doc_id를 넣고 전달하는 사용자만 바꿔 두 번째 조건이 결과에 미치는 영향을 따라갑니다.

1. 같은 문서 ID · document_id = 기존 초안의 ID
2. bob 조회 · id 일치 / owner 일치 = True / False
3. 행 없음 · get_title 반환값 = None


### 셀 01 · INSERT로 초안 저장

초기 자료는 커밋합니다. 이 셀을 다시 실행하면 모든 실습 값이 처음으로 돌아갑니다.


In [1]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE course_documents (id TEXT PRIMARY KEY, owner TEXT NOT NULL, title TEXT NOT NULL, content TEXT)")
doc_id = "54000000-0000-4000-8000-000000000001"
owner = "alice"
db.execute("INSERT INTO course_documents VALUES (?, ?, ?, ?)", (doc_id, owner, "초안", "SQL 변경 연습입니다."))
db.commit()

def get_title(user, document_id):
    row = db.execute("SELECT title FROM course_documents WHERE id = ? AND owner = ?", (document_id, user)).fetchone()
    return row[0] if row else None

print(get_title('alice', doc_id))

초안


초안이 보입니다. 문서 ID가 같아도 WHERE 조건의 owner가 다르면 같은 행을 읽을 수 없습니다. 실제 서비스의 owner는 인증을 마친 요청에서 정해야 합니다.

‘초안’이 조회됐다는 사실은 해당 행의 저장을 확인해 줍니다. 다른 사용자가 같은 문서 번호로 조회하거나 수정할 수 있는지는 별개입니다. 다음 쿼리에서 문서 번호와 소유자를 함께 조건에 넣어 접근 범위를 제한합니다.


### 문서 한 건을 담는 테이블

<p>course_documents의 id는 문서 한 건을 구별합니다. title과 content에는 각각 제목과 본문을 저장하고 owner에는 소유자를 저장합니다. 첫 셀이 초안을 넣고 commit한 뒤에는 이 행을 기준으로 수정 전후를 비교할 수 있습니다.</p>


### 테이블이 저장을 거부하는 값

<p>이 예제에서 id는 PRIMARY KEY이고 title과 owner에는 NOT NULL이 붙습니다. 같은 문서 ID를 다시 저장하거나 제목에 NULL을 넣으면 제약 위반입니다. owner 열에는 별도의 외래키가 선언되어 있지 않습니다.</p>


### 과제 02 · WHERE로 소유자 범위 제한

같은 문서 ID를 bob의 범위로 조회하세요. 행이 없을 때 함수가 반환하는 값을 표시합니다.


In [2]:
# get_title에 다른 사용자 ID를 전달하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(get_title(&#x27;bob&#x27;, doc_id))</code></pre>
<p>실행 결과</p><pre><code>None</code></pre>
<p>None입니다. 누락된 행을 정상적인 빈 결과로 다뤘습니다. 화면의 이름만 바꿔 다른 사용자의 문서를 보여 주지 않도록 조회 조건을 확인하세요.</p>
</details>


#### 수정할 행을 찾는 조건

<mark>문서 ID와 소유자 조건이 같은 행에서 모두 맞아야 합니다.</mark> 다른 소유자의 ID를 넣고 수정 전후의 값을 대조하세요. SQL이 오류 없이 끝났어도 조건에 맞는 행이 없으면 원하는 수정은 일어나지 않습니다.

선수 개념 · 53장 SQL SELECT·JOIN·GROUP BY

#### 같은 문서 ID로 소유자만 바꾼 두 조회

```mermaid
flowchart TD
nsame["같은 문서 ID"]
  nalice["alice 조회"]
  ntitle["제목 반환"]
  nbob["bob 조회"]
  nnone["행 없음"]
  nsame --> nalice
  nalice --> ntitle
  nsame --> nbob
  nbob --> nnone
```

get_title에 같은 doc_id를 넣고 전달하는 사용자만 바꿔 두 번째 조건이 결과에 미치는 영향을 따라갑니다.

### 문서 ID와 소유자 두 조건의 효과

<p>53장의 사용자 필터에 문서 ID를 더했습니다. 같은 doc_id로 조회해도 bob에게는 None이 돌아옵니다.</p>


## UPDATE와 rollback()

트랜잭션은 함께 확정하거나 되돌리는 변경의 단위입니다. UPDATE 직후 같은 연결에서는 바뀐 제목을 읽을 수 있습니다. 커밋 전에 롤백하면 앞서 커밋한 초안으로 돌아갑니다.


### 셀 03 · UPDATE 직후 행 조회

아직 commit을 호출하지 않습니다.


In [3]:
db.execute("UPDATE course_documents SET title = ? WHERE id = ? AND owner = ?", ("수정한 초안", doc_id, owner))
print(get_title(owner, doc_id))

수정한 초안


수정한 초안입니다. 이 출력은 같은 연결에서 새 값을 읽었다는 증거이며 다른 연결에서도 이 변경을 읽을 수 있는지는 보여 주지 않습니다.


### 문서를 새로 넣기와 제목 바꾸기

<p>INSERT는 새 ID와 소유자·제목·본문을 한 행으로 저장합니다. 뒤의 UPDATE는 WHERE에서 같은 문서 ID와 소유자를 모두 지정하고, 이미 있는 행의 title만 수정한 초안으로 바꿉니다.</p>


### 셀 04 · rollback()으로 변경 취소

위 셀 직후 rollback을 호출해 처음의 행을 다시 조회합니다.


In [4]:
db.rollback()
print(get_title(owner, doc_id))

초안


초안이 복구됩니다. UPDATE 직후에 commit을 넣으면 이어지는 rollback은 이미 확정한 변경을 되돌리지 못합니다.


### 마지막 커밋으로 돌아가기

<p>초안을 INSERT한 뒤 한 번 commit했습니다. 이어 UPDATE하면 같은 연결의 SELECT에는 수정한 초안이 보입니다. 여기서 rollback하면 마지막 커밋 이후의 UPDATE만 취소되어 문서 행은 남고 제목이 초안으로 돌아갑니다.</p>


### 과제 05 · DELETE의 조건 범위

alice가 가진 이 문서 한 건만 지운 뒤 조회하고, 연습이 끝나면 rollback으로 복원하세요.

삭제한 같은 행을 두 번 조회하세요. 첫 조회는 DELETE 직후, 두 번째는 rollback() 직후입니다. 문서 제목이 돌아왔는지는 두 번째 출력에서 확인합니다.


In [5]:
# DELETE에서 id와 owner를 모두 검사하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>db.execute(&quot;DELETE FROM course_documents WHERE id = ? AND owner = ?&quot;, (doc_id, owner))
print(get_title(owner, doc_id))
db.rollback()
print(get_title(owner, doc_id))</code></pre>
<p>실행 결과</p><pre><code>None
초안</code></pre>
<p>삭제 직후 첫 조회에는 행이 없습니다. 롤백 뒤 같은 소유자와 같은 ID로 다시 조회하면 원래 제목 ‘초안’이 보입니다.</p>
</details>


### DELETE와 대상 행 검사

<p>DELETE는 조건에 맞는 행을 지웁니다. 열 하나를 비우는 UPDATE와 다릅니다. 아래 문장은 오늘 만든 새 실습 문서 한 행만 대상으로 삼습니다. WHERE가 빠진 DELETE는 테이블의 모든 행을 대상으로 하므로 그대로 실행하지 않습니다.</p>

<p>셀 05에서는 삭제를 커밋하지 않습니다. 삭제 직후 get_title은 None을 반환하고 rollback 다음에는 앞서 확정한 제목 ‘초안’을 다시 읽습니다.</p>

<table><caption>세 명령 비교</caption><thead><tr><th scope="col">명령</th><th scope="col">바뀌는 것</th><th scope="col">실행 뒤 확인</th></tr>
</thead><tbody><tr><td>INSERT</td><td>새 행 추가</td><td>같은 ID 조회</td></tr>
<tr><td>UPDATE</td><td>조건에 맞는 행의 열 값</td><td>제목과 변경 행 수</td></tr>
<tr><td>DELETE</td><td>조건에 맞는 행 제거</td><td>같은 ID 조회가 0행인지</td></tr>
</tbody></table>


## 선택 실습 · PostgreSQL 값 바인딩

준비한 커널에서는 psycopg로 PostgreSQL의 임시 연습 표를 만듭니다. 실행 중 만든 표와 행은 트랜잭션 끝에 롤백합니다. 기본 연결은 53장에서 강사가 설정한 수업용 DB를 사용합니다.


### 셀 06 · 선택 실습 · 매개변수 바인딩

HK_COURSE_USE_POSTGRES=1과 수업용 연결이 준비되면 아래 블록이 실행됩니다. 저장소의 영구 테이블에는 쓰지 않습니다.


In [6]:
from importlib.util import find_spec
import os
pg_ready = find_spec("psycopg") is not None and os.getenv("HK_COURSE_USE_POSTGRES") == "1"
if pg_ready:
    import psycopg
    with psycopg.connect(connect_timeout=3) as conn:
        conn.execute("CREATE TEMP TABLE hk_note (id integer PRIMARY KEY, title text NOT NULL)")
        conn.execute("INSERT INTO hk_note VALUES (%s, %s)", (1, "조회문 복습"))
        value = conn.execute("SELECT title FROM hk_note WHERE id = %s", (1,)).fetchone()
        print(value[0])
        conn.rollback()
else:
    print("PostgreSQL 변경은 준비된 수업 커널에서 확인하세요.")

PostgreSQL 변경은 준비된 수업 커널에서 확인하세요.


PostgreSQL 선택 셀에서 임시 표를 만든 경우 출력은 ‘조회문 복습’입니다. 접속 준비 안내는 PostgreSQL에서 변경이 성공했다는 뜻이 아닙니다. 앞에서 실행한 메모리 실습은 SQL 변경 순서를 확인한 결과입니다. 서버 실습에는 psycopg, 수업용 PostgreSQL 연결, 임시 테이블 생성 권한이 필요합니다. HK_COURSE_USE_POSTGRES=1인 수업 커널에서 실행한 뒤 바인딩한 제목과 다시 읽은 값을 비교하세요.


### 선택 셀의 PostgreSQL 연결

<p>이 셀은 psycopg가 설치되고 HK_COURSE_USE_POSTGRES 값이 1일 때 연결을 엽니다. 연결이 준비되면 임시 테이블 hk_note에서 값을 바인딩해 저장·조회하고 rollback합니다. 기본 SQLite 셀은 PostgreSQL 연결 없이도 실행할 수 있습니다.</p>


### SQLite의 ?와 Psycopg의 %s

<p>기본 셀의 sqlite3 쿼리는 값 자리에 ?를 씁니다. PostgreSQL 선택 셀에서는 psycopg가 %s를 사용합니다. 두 경우 모두 SQL과 전달할 값을 execute의 별도 인수로 넘깁니다. 자리 표시자가 달라도 문자열을 붙여 SQL을 만드는 과정은 없습니다.</p>


## 복습과 실습 점검

### 과제 07 · 수정된 행이 0건일 때의 결과 확인하기

문서 ID는 그대로 두고 bob이 제목을 바꾸려는 경우를 조회하세요. UPDATE로 바뀐 행 수와 alice가 다시 읽은 제목을 함께 출력하세요.


In [ ]:
# 제목을 '확정본'으로 바꾸되 id와 owner를 모두 조건에 넣으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>change = db.execute(
    &quot;UPDATE course_documents SET title = ? WHERE id = ? AND owner = ?&quot;,
    (&quot;확정본&quot;, doc_id, &quot;bob&quot;),
)
print(change.rowcount, get_title(&quot;alice&quot;, doc_id))
db.rollback()</code></pre>
<p>실행 결과</p><pre><code>0 초안</code></pre>
<p>SQL 문장은 정상 실행됐지만 두 조건을 만족한 행이 없습니다. 성공 응답을 만들기 전에 반영된 행 수를 확인하면 엉뚱한 사용자에게 수정 완료를 알리는 일을 막을 수 있습니다.</p>
</details>


## 수정 내용을 저장할지 되돌릴지 정하기

학습자가 자기 문서의 제목을 수정합니다. 페이지가 열려 있던 사이 다른 사람이 같은 계정에서 제목을 먼저 바꿨다면 새 제목을 덮어쓰지 않도록 해 봅니다. 로컬 연습 DB에서 작은 트랜잭션을 실행합니다.

### 셀 09 · 수정하기 전에 알고 있던 제목 확인하기

전용 문서 edit-54의 현재 제목을 검토 중으로 준비합니다. 화면은 이전 제목인 초안을 기억합니다. 이 값으로 변경하면 몇 행이 바뀔까요?

In [7]:
edit_id = "edit-54"
db.execute("DELETE FROM course_documents WHERE id = ?", (edit_id,))
db.execute("INSERT INTO course_documents VALUES (?, ?, ?, ?)",
           (edit_id, "alice", "검토 중", "편집 연습"))
db.commit()
rename_sql = """
UPDATE course_documents SET title = ?
WHERE id = ? AND owner = ? AND title = ?
"""
attempt = db.execute(rename_sql, ("확정본", edit_id, "alice", "초안"))
print(attempt.rowcount, get_title("alice", edit_id))
db.rollback()

0 검토 중


0 검토 중이 나옵니다. id와 owner는 맞았지만 현재 행의 title이 초안과 달랐습니다. 쿼리 실행 자체는 성공했으므로 화면은 rowcount로 수정 성공 여부를 확인해야 합니다.

### 과제 10 · 일치하는 제목만 고치고 커밋하기

최신 제목을 확인한 alice가 다시 시도합니다. rename_sql에 검토 중을 이전 제목으로 넘기고, 한 행이 바뀌면 커밋하세요. 다른 결과면 되돌리세요. 끝에 실제 저장된 제목을 읽으세요.

In [8]:
# rename_sql에 새 제목, ID, 소유자, 읽었던 제목 순서로 바인딩하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>changed = db.execute(rename_sql, (&quot;확정본&quot;, edit_id, &quot;alice&quot;, &quot;검토 중&quot;))
if changed.rowcount == 1:
    db.commit()
else:
    db.rollback()
print(changed.rowcount, get_title(&quot;alice&quot;, edit_id))</code></pre>
<p>실행 결과</p><pre><code>1 확정본</code></pre>
<p>1 확정본이 출력됩니다. 방금 읽은 제목이 조건과 맞았고 커밋 뒤 조회에서도 새 값이 보입니다. 이 과제를 다시 풀 때는 바로 위 제공 셀부터 실행해 검토 중 상태로 돌리세요.</p>
</details>

<mark>UPDATE가 오류 없이 끝나도 바뀐 행은 0개일 수 있습니다.</mark> 이전 제목까지 WHERE에 넣으면 다른 편집 결과 위에 내 변경을 조용히 덮어쓰는 일을 피할 수 있습니다.

서비스의 수정 함수는 변경 건수로 충돌을 알리고, 화면은 최신 문서를 다시 읽을 수 있습니다.

## UPDATE와 commit으로 수정한 규정 넘기기

53장에서 조회한 한 행을 작은 테이블에 다시 넣습니다. 어떤 소유자의 문서가 바뀌었는지 확인하고 새 문장을 저장하세요.

In [ ]:
import sqlite3
record = previous["record"].copy()
desk_db = sqlite3.connect(":memory:")
desk_db.execute("CREATE TABLE rules (owner TEXT, id TEXT, text TEXT, PRIMARY KEY(owner,id))")
desk_db.execute("INSERT INTO rules VALUES (:owner, :id, :text)", record)
revised_days = 14
update_sql = "UPDATE rules SET text=? WHERE owner=? AND id=?"
new_text = f"대출 기간은 {revised_days}일입니다."
with desk_db:
    count = desk_db.execute(update_sql, (new_text, record["owner"], record["id"])).rowcount
record["text"] = desk_db.execute("SELECT text FROM rules WHERE owner=? AND id=?",
                                 (record["owner"], record["id"])).fetchone()[0]
print(count, record["text"])

1 대출 기간은 14일입니다.


기본 자료에서는 1건이 바뀌고 14일이 나옵니다. with 블록이 정상적으로 끝나야 수정이 커밋됩니다. revised_days를 21로 바꾸고 이 셀부터 다시 실행하면 다음 수업도 21일로 시작합니다.

### 직접 확인하기

같은 UPDATE에 소유자만 bob으로 바꿔 넣으세요. 갱신한 행 수가 0일 때 기존 문장이 남는지 확인합니다.

In [ ]:
# bob의 수정 요청을 실행하고 rowcount와 record의 문장을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary><pre><code>with desk_db:
    changed = desk_db.execute(update_sql, (&quot;미확인&quot;, &quot;bob&quot;, record[&quot;id&quot;])).rowcount
stored = desk_db.execute(&quot;SELECT text FROM rules WHERE owner=? AND id=?&quot;,
                         (record[&quot;owner&quot;], record[&quot;id&quot;])).fetchone()[0]
print(changed, stored)</code></pre><p>기본 자료로 실행한 결과</p><pre><code>0 대출 기간은 14일입니다.</code></pre><p>bob의 행이 없어 UPDATE가 0건을 돌려줍니다. record의 소유자와 번호로 다시 읽으면 실제로 유지된 문장을 확인할 수 있습니다.</p></details>

### 55장에 넘길 결과 저장하기

55장에서 이 행을 도구의 실제 반환값으로 씁니다. 저장 셀을 다시 실행하면 같은 파일에 이번 결과를 기록합니다.

In [ ]:
product = {"record": record}
target = work / "agent-54.json"
target.write_text(json.dumps(product, ensure_ascii=False, indent=2), encoding="utf-8")
print(target.name)

agent-54.json
